<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/Building_a_trend_following_strategy_and_comparing_with_Backtrader_and_VectorBT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Building a trend following strategy and comparing with Backtrader and VectorBT
# https://medium.com/@jiahau3/building-a-trend-following-strategy-and-comparing-with-backtrader-and-vectorbt-563c64fbbc76
# https://www.youtube.com/watch?v=el0V-3Gb2rc

In [2]:
!pip install websockets

In [3]:
!pip install yfinance

import sqlalchemy
import pandas as pd
import yfinance as yf # Import yfinance for historical data

# Removed websockets, json, asyncio imports and related Binance live data streaming code.

# Define a function to fetch historical data from Yahoo Finance
def get_yfinance_data(symbol, start_date, end_date=None, interval='1d'):
    """
    Fetches historical stock data from Yahoo Finance.

    Args:
        symbol (str): The ticker symbol (e.g., 'AAPL', 'SPY').
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str, optional): End date in 'YYYY-MM-DD' format. Defaults to None (today).
        interval (str, optional): Data interval (e.g., '1d', '1wk', '1mo'). Defaults to '1d'.

    Returns:
        pd.DataFrame: Historical data with 'Open', 'High', 'Low', 'Close', 'Volume' columns.
    """
    try:
        print(f"Downloading data for {symbol} from {start_date} to {end_date}...")
        data = yf.download(symbol, start=start_date, end=end_date, interval=interval)

        if data.empty:
            print(f"No data found for {symbol} for the specified period after download attempt.")
            return pd.DataFrame()

        print(f"DEBUG: Raw columns from yf.download: {data.columns}") # DEBUG print

        data.index.name = 'Time'

        # Handle potential MultiIndex columns (e.g., if yfinance returns ( 'Close', 'SPY'))
        # The goal is to get 'Close', 'High', 'Low', 'Open', 'Volume'
        if isinstance(data.columns, pd.MultiIndex):
            print("DEBUG: MultiIndex detected, flattening to first level names.") # DEBUG print
            # Extract the first level values (e.g., 'Open', 'High', 'Low', 'Close', 'Volume')
            data.columns = data.columns.get_level_values(0)

        # Ensure column names are capitalized (e.g., 'open' -> 'Open')
        data.columns = [col.capitalize() for col in data.columns]

        print(f"DEBUG: Columns after processing: {data.columns}") # DEBUG print

        # Select the required OHLCV columns. This ensures consistency.
        required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

        # Check if all required columns are present after processing
        if not all(col in data.columns for col in required_cols):
            missing_cols = [col for col in required_cols if col not in data.columns]
            print(f"Error: Missing required columns in data for {symbol}: {missing_cols}")
            return pd.DataFrame()

        data = data[required_cols]
        return data
    except Exception as e:
        print(f"Error fetching data for {symbol}: {e}")
        return pd.DataFrame()

# Example usage for backtesting
if __name__ == "__main__":
    # You can change the symbol and dates as needed for your backtest
    backtest_symbol = 'SPY' # Example: S&P 500 ETF
    backtest_start_date = '2020-01-01'
    backtest_end_date = '2023-01-01'

    df_yfinance = get_yfinance_data(backtest_symbol, backtest_start_date, backtest_end_date)

    if not df_yfinance.empty:
        print(f"Successfully fetched {len(df_yfinance)} rows of data for {backtest_symbol}.")
        print("First 5 rows:")
        print(df_yfinance.head())
        print("\nLast 5 rows:")
        print(df_yfinance.tail())

        # Store this data to an SQLite database
        dbfile_yfinance = 'yfinance_data.db'
        engine_yfinance = sqlalchemy.create_engine(f'sqlite:///{dbfile_yfinance}')
        df_yfinance.to_sql(backtest_symbol.lower(), engine_yfinance, if_exists='replace', index=True)
        print(f"\nData stored to {dbfile_yfinance} in table '{backtest_symbol.lower()}'.")
    else:
        print("Failed to fetch data or data is empty.")


/tmp/ipykernel_6627/3009335545.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date, interval=interval)
[*********************100%***********************]  1 of 1 completed

DEBUG: Raw columns from yf.download: MultiIndex([( 'Close', 'SPY'),
            (  'High', 'SPY'),
            (   'Low', 'SPY'),
            (  'Open', 'SPY'),
            ('Volume', 'SPY')],
           names=['Price', 'Ticker'])
DEBUG: MultiIndex detected, flattening to first level names.
DEBUG: Columns after processing: Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Successfully fetched 756 rows of data for SPY.
First 5 rows:
                  Open        High         Low       Close    Volume
Time                                                                
2020-01-02  295.672752  296.906479  294.749737  296.888184  59151200
2020-01-03  293.497680  295.764082  293.442850  294.640015  77709700
2020-01-06  292.885455  295.846405  292.766647  295.764160  55653900
2020-01-07  295.197466  295.672695  294.484651  294.932465  40496400
2020-01-08  295.124476  297.719857  294.877741  296.504425  68296000

Last 5 rows:
                  Open        High         Low     

In [4]:
pip install python-binance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.5/148.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 27.4 MB/s eta 0:00:00


In [5]:
import pandas as pd
from binance.client import Client
import sqlalchemy

def getSymbolpairs(quote_symbol):
    client = Client()
    info = client.get_exchange_info()
    # Select the token pairs end with BUSD
    symbols = [x['symbol'] for x in info['symbols'] if x['symbol'].endswith(quote_symbol)]
    return symbols

def getPricedata(symbol: str, kline_interval: str, date: str):
    client = Client()
    df = pd.DataFrame(client.get_historical_klines(symbol, kline_interval, date))
    if len(df) > 0:
        df = df.iloc[:,[0,1,2,3,4,5,9]]
        df.columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Taker_buy_volume']
        df.Time = pd.to_datetime(df.Time, unit='ms')
        df = df.set_index('Time')
        df = df.astype(float)
        return df

def createDataframe(symbols):
    symbol_price = {}
    for symbol in symbols:
        df = getPricedata(symbol, '1d', '2019-01-01')
        if df is not None:
            symbol_price[symbol] = df
    symbol_price = pd.concat(symbol_price).reset_index()
    symbol_price = symbol_price.rename(columns={'level_0': 'Symbol'})
    return symbol_price

def to_database(symbol_price, tablename, dbfile: str):
    engine = sqlalchemy.create_engine('sqlite:///' + dbfile)
    symbol_price.to_sql(tablename, engine, index=False)
    print(f'Data imported to {dbfile}')
    return -1

def create_cryptoList():

    pass

if __name__ == '__main__':
    tokens = getSymbolpairs('BUSD')
    tokens_price = createDataframe(tokens)
    to_database(tokens_price, 'crypto_price', 'BUSDquote.db')

BinanceAPIException: APIError(code=0): Service unavailable from a restricted location according to 'b. Eligibility' in https://www.binance.com/en/terms. Please contact customer service if you believe you received this message in error.

In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import config

plt.style.use('classic')
plt.rcParams['figure.figsize'] = (15,10)
color_pal = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
engine = sqlalchemy.create_engine(config.BUSD_DB_URL)
df = pd.read_sql(f'SELECT * from {config.BUSD_DB_PRICE_TABLE}', engine)

In [ ]:
df.head()

In [ ]:
df.set_index(pd.to_datetime(df.Time)).query('Symbol == "BTCBUSD"').plot()

In [ ]:
price_btc = df.set_index(pd.to_datetime(df.Time)).\
    query('Symbol == "BTCBUSD"')[['Volume','Close']]

In [ ]:
dailyreturn = df[['Time','Symbol','Close']].\
    pivot(index='Time', columns='Symbol', values='Close').\
    dropna(axis=1).\
    pct_change()
dailyreturn.index = pd.to_datetime(dailyreturn.index)
dailyreturn

In [ ]:
dailyreturn['BTCBUSD'].plot()

In [ ]:
fig, axs = plt.subplots(2,1)
price_btc.plot(ax=axs[0])
dailyreturn['BTCBUSD'].plot(ax=axs[1])
axs[1].axhline(y=0, color='r')

# SMA cross over strategy
It is based on two smooth moving average(SMA) price value to create trading signals. If fast line is crossing above slow one, the buy signal is created. Vice versa to the sell signal.

In [ ]:
price_btc['close_sma10'] = price_btc['Close'].rolling(10).mean()
price_btc['close_sma20'] = price_btc['Close'].rolling(20).mean()
price_btc['close_sma50'] = price_btc['Close'].rolling(50).mean()
price_btc['volume_sma20'] = price_btc['Volume'].rolling(20).mean()

In [ ]:
price_btc['c10_High'] = np.where(price_btc['close_sma10'] > price_btc['close_sma20'], 1, 0)
price_btc['signals'] = np.where((price_btc['c10_High'] - price_btc['c10_High'].shift(1)) == 1, 1, 0)
price_btc['signals'] = np.where((price_btc['c10_High'] - price_btc['c10_High'].shift(1)) == -1, -1, price_btc['signals'])

stoploss = 0.1
position = np.zeros(price_btc.shape[0])
for i in range(price_btc.shape[0]):
    if price_btc['signals'][i] == 1:
        position[i:] += 1
    if (price_btc['signals'][i] == -1) & (position[i] == 1):
        position[i:] -= 1

# Stop loss
#
price_btc['return'] = price_btc['Close'].pct_change()
price_btc['position'] = pd.Series(position, index=price_btc.index).shift(1)
price_btc['strategy_return'] = price_btc['return'] * price_btc['position']
# price_btc['sell_signals'] = np.where(price_btc['close_sma10'] < price_btc['close_sma20'], 1, 0)

In [ ]:
price_btc.tail()

In [ ]:
price_btc[['Close','close_sma10','close_sma20','close_sma50','volume_sma20']].plot(figsize=(20,8))
plt.scatter(price_btc.index[price_btc['signals'] == 1],\
     price_btc.loc[price_btc.index[price_btc['signals'] == 1]]['Close'],\
     marker='^',color=color_pal[color_pal.index('g')], s=50)
plt.scatter(price_btc.index[price_btc['signals'] == -1],\
     price_btc.loc[price_btc.index[price_btc['signals'] == -1]]['Close'],\
     marker='v',color=color_pal[color_pal.index('r')], s=50)

In [ ]:
print(((price_btc['return']+1).cumprod()-1)[-1])
print(((price_btc['strategy_return']+1).cumprod()-1)[-1])

In [ ]:
_, axs = plt.subplots(1,1)
((price_btc['return']+1).cumprod()-1).plot(ax=axs, legend='buy_hold')
((price_btc['strategy_return']+1).cumprod()-1).plot(ax=axs, legend='sma')

In [ ]:
import vectorbt as vbt

price = price_btc['Close']
pf = vbt.Portfolio.from_holding(price, init_cash=100)
pf.total_profit()

In [ ]:
fast_ma = vbt.MA.run(price, 10)
slow_ma = vbt.MA.run(price, 20)
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

pf = vbt.Portfolio.from_signals(price, entries, exits, init_cash=100)
pf.total_profit()

In [ ]:
pf.plot()

In [ ]:
import backtrader as bt

In [ ]:
class smaCross(bt.SignalStrategy):
    def __init__(self):
        sma1, sma2 = bt.ind.SMA(period=10), bt.ind.SMA(period=20)
        crossover = bt.ind.CrossOver(sma1, sma2)
        self.signal_add(bt.SIGNAL_LONG, crossover)


In [ ]:
cerebro0 = bt.Cerebro()
cerebro0.addstrategy(smaCross)

cerebro0.adddata(bt.feeds.PandasData(dataname = price_btc[['Close', 'Volume']].iloc[:],\
     close='Close', volume='Volume', open='Close', high='Close', low='Close'))

cerebro0.broker.set_cash(100)
cerebro0.addsizer(bt.sizers.PercentSizer, percents=100)

print('Starting Portfolio Value: %.2f' % cerebro0.broker.get_value())
cerebro0.run()
cerebro0.plot()
print('Final Portfolio Value: %.2f' % cerebro0.broker.get_value())

In [ ]:
class smaCro(bt.Strategy):
    params = dict(
        pfast=10,
        pslow=20
    )

    def log(self, txt, dt=None):
        ''' Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        sma1 = bt.ind.SMA(period=self.params.pfast)
        sma2 = bt.ind.SMA(period=self.params.pslow)
        self.crossover = bt.ind.CrossOver(sma1, sma2)

    def start(self):
        self.val_start = self.broker.get_cash()  # keep the starting cash

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
                # self.log('Buy create: %.2f' % self.data.close[0])
        else:
            if self.crossover < 0:
                self.sell()
                # self.log('Sell create: %.2f' % self.data.close[0])

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(
                    'BUY EXECUTED, Price: %.5f, Cost: %.f, Comm %.2f' %
                    (order.executed.price,
                     order.executed.value,
                     order.executed.comm))

                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            else:  # Sell
                self.log('SELL EXECUTED, Price: %.5f, Cost: %.f, Comm %.2f' %
                         (order.executed.price,
                          order.executed.value,
                          order.executed.comm))

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        self.order = None

    def stop(self):
        # calculate the actual returns
        self.roi = (self.broker.get_value() / self.val_start) - 1.0
        print('ROI:        {:.2f}%'.format(100.0 * self.roi))

In [ ]:
cerebro = bt.Cerebro()
cerebro.addstrategy(smaCro)

cerebro.adddata(bt.feeds.PandasData(dataname = price_btc[['Close', 'Volume']].iloc[:],\
     close='Close', volume='Volume', open='Close', high='Close', low='Close'))

cerebro.broker.set_cash(100)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

print('Starting Portfolio Value: %.2f' % cerebro.broker.get_value())
cerebro.run()
cerebro.plot()
print('Final Portfolio Value: %.2f' % cerebro.broker.get_value())

In [ ]:
class BuyAndHold_1(bt.Strategy):
    def start(self):
        self.val_start = self.broker.get_cash()  # keep the starting cash

    def nextstart(self):
        # Buy all the available cash
        size = int(self.broker.get_cash())
        self.buy()

    def stop(self):
        # calculate the actual returns
        self.roi = (self.broker.get_value() / self.val_start) - 1.0
        print('ROI:        {:.2f}%'.format(100.0 * self.roi))

In [ ]:
cerebroH = bt.Cerebro()
cerebroH.addstrategy(BuyAndHold_1)

cerebroH.adddata(bt.feeds.PandasData(dataname = price_btc[['Close', 'Volume']].iloc[:],\
     close='Close', volume='Volume', open='Close', high='Close', low='Close'))

cerebroH.broker.set_cash(100)
cerebroH.addsizer(bt.sizers.PercentSizer, percents=100)

print('Starting Portfolio Value: %.2f' % cerebroH.broker.get_value())
cerebroH.run()
cerebroH.plot()
print('Final Portfolio Value: %.2f' % cerebroH.broker.get_value())